# Results Interpretation Document

Converted from the original Python workflow script so the dissertation repository uses notebook-based workflow artefacts.


In [ ]:
from pathlib import Path
import csv
import json

from docx import Document
from docx.enum.table import WD_ALIGN_VERTICAL
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from docx.shared import Inches, Pt, RGBColor


ROOT = Path("/Users/lu_nanxi/CASA/Dissertation_Data")
MODEL_DIR = ROOT / "Vivacity_full_day_cutoff_20260526" / "modelling_ready" / "models"
GATE_DIR = ROOT / "Vivacity_full_day_cutoff_20260526" / "integrity_gate"
OUT = ROOT / "dissertation_results_vivacity_exploratory_analysis.docx"


def set_cell_shading(cell, fill):
    tc_pr = cell._tc.get_or_add_tcPr()
    shd = OxmlElement("w:shd")
    shd.set(qn("w:fill"), fill)
    tc_pr.append(shd)


def set_cell_margins(cell, top=80, start=120, bottom=80, end=120):
    tc = cell._tc
    tc_pr = tc.get_or_add_tcPr()
    tc_mar = tc_pr.first_child_found_in("w:tcMar")
    if tc_mar is None:
        tc_mar = OxmlElement("w:tcMar")
        tc_pr.append(tc_mar)
    for m, v in [("top", top), ("start", start), ("bottom", bottom), ("end", end)]:
        node = tc_mar.find(qn(f"w:{m}"))
        if node is None:
            node = OxmlElement(f"w:{m}")
            tc_mar.append(node)
        node.set(qn("w:w"), str(v))
        node.set(qn("w:type"), "dxa")


def set_table_borders(table):
    tbl_pr = table._tbl.tblPr
    borders = tbl_pr.first_child_found_in("w:tblBorders")
    if borders is None:
        borders = OxmlElement("w:tblBorders")
        tbl_pr.append(borders)
    for edge in ("top", "left", "bottom", "right", "insideH", "insideV"):
        elem = borders.find(qn(f"w:{edge}"))
        if elem is None:
            elem = OxmlElement(f"w:{edge}")
            borders.append(elem)
        elem.set(qn("w:val"), "single")
        elem.set(qn("w:sz"), "4")
        elem.set(qn("w:space"), "0")
        elem.set(qn("w:color"), "DADCE0")


def add_run(paragraph, text, bold=False, size=11, color="000000"):
    run = paragraph.add_run(text)
    run.bold = bold
    run.font.name = "Arial"
    run._element.rPr.rFonts.set(qn("w:eastAsia"), "Arial")
    run.font.size = Pt(size)
    run.font.color.rgb = RGBColor.from_string(color)
    return run


def add_para(doc, text):
    p = doc.add_paragraph()
    p.paragraph_format.space_after = Pt(8)
    p.paragraph_format.line_spacing = 1.15
    add_run(p, text)
    return p


def add_heading(doc, text, level=1):
    p = doc.add_paragraph()
    p.paragraph_format.space_before = Pt(20 if level == 1 else 14)
    p.paragraph_format.space_after = Pt(6)
    add_run(p, text, bold=False, size=20 if level == 1 else 15, color="000000")
    return p


def add_bullets(doc, items):
    for text in items:
        p = doc.add_paragraph(style="List Bullet")
        p.paragraph_format.space_after = Pt(4)
        p.paragraph_format.line_spacing = 1.15
        add_run(p, text)


def read_csv(path):
    with path.open(newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))


def pct(x):
    return f"{float(x):+.2f}%"


def pval(x):
    v = float(x)
    if v < 0.001:
        return "<0.001"
    return f"{v:.3f}"


def label_direction(direction):
    labels = {
        "negative_statistically_flagged": "negative, statistically flagged",
        "positive_statistically_flagged": "positive, statistically flagged",
        "negative_uncertain": "negative, uncertain",
        "positive_uncertain": "positive, uncertain",
    }
    return labels.get(direction, direction.replace("_", " "))


def split_headlines(rows):
    out = {}
    for row in rows:
        key = (row["scheme_id"], row["outcome_label"])
        out.setdefault(key, {})[row["term"]] = row
    return out


def add_headline_table(doc, headline_rows):
    grouped = split_headlines(headline_rows)
    table = doc.add_table(rows=1, cols=6)
    table.autofit = False
    widths = [0.62, 1.05, 1.35, 0.72, 1.35, 1.38]
    for i, width in enumerate(widths):
        table.columns[i].width = Inches(width)
    set_table_borders(table)
    headers = [
        "Scheme",
        "Outcome",
        "Immediate change",
        "p",
        "Weekly slope change",
        "Interpretation",
    ]
    for i, header in enumerate(headers):
        cell = table.rows[0].cells[i]
        set_cell_shading(cell, "F1F3F4")
        set_cell_margins(cell)
        p = cell.paragraphs[0]
        add_run(p, header, bold=True, size=9)

    for scheme in ["12d", "12f", "13"]:
        for outcome in ["Active travel", "Pedestrian", "Cyclist"]:
            pair = grouped[(scheme, outcome)]
            level = pair["treated_postTRUE"]
            slope = pair["treated_post_time_weeks"]
            cells = table.add_row().cells
            values = [
                scheme,
                outcome,
                pct(level["percent_change"]),
                pval(level["p_value"]),
                f"{pct(slope['percent_change'])} per week (p={pval(slope['p_value'])})",
                label_direction(slope["direction"])
                if "statistically_flagged" in slope["direction"]
                else label_direction(level["direction"]),
            ]
            for i, value in enumerate(values):
                set_cell_margins(cells[i])
                cells[i].vertical_alignment = WD_ALIGN_VERTICAL.CENTER
                p = cells[i].paragraphs[0]
                p.paragraph_format.space_after = Pt(0)
                if i in [0, 3]:
                    p.alignment = WD_ALIGN_PARAGRAPH.CENTER
                add_run(p, value, size=8.5)


def add_fit_table(doc, fit_rows):
    by_scheme = {}
    for row in fit_rows:
        by_scheme.setdefault(row["scheme_id"], row)
    table = doc.add_table(rows=1, cols=7)
    table.autofit = False
    widths = [0.75, 1.25, 1.05, 1.05, 1.05, 1.1, 1.1]
    for i, width in enumerate(widths):
        table.columns[i].width = Inches(width)
    set_table_borders(table)
    headers = [
        "Scheme",
        "Date range",
        "Rows",
        "Countlines",
        "Treated",
        "Controls",
        "Intervention",
    ]
    for i, header in enumerate(headers):
        cell = table.rows[0].cells[i]
        set_cell_shading(cell, "F1F3F4")
        set_cell_margins(cell)
        add_run(cell.paragraphs[0], header, bold=True, size=9)
    for scheme in ["12d", "12f", "13"]:
        row = by_scheme[scheme]
        cells = table.add_row().cells
        values = [
            scheme,
            f"{row['first_week']} to {row['last_week']}",
            row["n_rows"],
            row["n_countlines"],
            row["n_treated_countlines"],
            row["n_control_countlines"],
            row["intervention_date"],
        ]
        for i, value in enumerate(values):
            set_cell_margins(cells[i])
            cells[i].vertical_alignment = WD_ALIGN_VERTICAL.CENTER
            p = cells[i].paragraphs[0]
            p.paragraph_format.space_after = Pt(0)
            if i != 1:
                p.alignment = WD_ALIGN_PARAGRAPH.CENTER
            add_run(p, value, size=8.5)


def add_plot(doc, path, caption):
    if not path.exists():
        return
    p = doc.add_paragraph()
    p.paragraph_format.space_before = Pt(12)
    p.paragraph_format.space_after = Pt(4)
    add_run(p, caption, bold=True, size=10)
    doc.add_picture(str(path), width=Inches(6.35))


def build_doc():
    headline_rows = read_csv(MODEL_DIR / "vivacity_exploratory_headline_effects.csv")
    fit_rows = read_csv(MODEL_DIR / "vivacity_exploratory_model_fit_stats.csv")
    with (GATE_DIR / "vivacity_integrity_gate_summary.json").open(encoding="utf-8") as f:
        gate = json.load(f)

    doc = Document()
    section = doc.sections[0]
    section.top_margin = Inches(1)
    section.bottom_margin = Inches(1)
    section.left_margin = Inches(1)
    section.right_margin = Inches(1)

    styles = doc.styles
    styles["Normal"].font.name = "Arial"
    styles["Normal"]._element.rPr.rFonts.set(qn("w:eastAsia"), "Arial")
    styles["Normal"].font.size = Pt(11)

    title = doc.add_paragraph()
    title.paragraph_format.space_after = Pt(3)
    add_run(title, "Results Draft: Vivacity Exploratory Models", size=26)

    subtitle = doc.add_paragraph()
    subtitle.paragraph_format.space_after = Pt(12)
    add_run(
        subtitle,
        "Current interpretation for dissertation writing and discussion",
        size=11,
        color="555555",
    )

    add_heading(doc, "Headline Conclusion", 1)
    add_para(
        doc,
        "The current Vivacity evidence does not yet support a strong causal claim that the analysed active travel schemes produced clear increases in walking or cycling relative to matched controls. The exploratory models are more consistent with mixed and uncertain relative trajectories, with two negative slope terms statistically flagged: cyclist counts for scheme 12d and active travel counts for scheme 12f.",
    )
    add_para(
        doc,
        "This should be written cautiously. The strongest dissertation wording is that the analysis tests whether treated sites diverged from matched non-intervention countlines, but the current evidence should be interpreted as exploratory because the matched-control pool is limited and concurrent intervention exposure at control locations requires careful interpretation.",
    )

    add_heading(doc, "Analysis Base", 1)
    add_bullets(
        doc,
        [
            f"The integrity gate covered {gate['all_countlines']} countlines: {gate['treated_countlines']} treated and {gate['control_countlines']} control countlines.",
            f"Thirteen controls met the pragmatic causal threshold; two met the stricter seasonal threshold.",
            "The final exploratory models were fitted only for schemes 12d, 12f, and 13.",
            "Schemes 12b and 12e are retained for descriptive analysis because no quality-passing pre-installation treated observations remain.",
            "Confirmed intervention dates are encoded in the current workflow.",
        ],
    )
    add_fit_table(doc, fit_rows)

    add_heading(doc, "Model Results", 1)
    add_para(
        doc,
        "The table reports the two treated-control terms that matter most for interpretation: the immediate post-intervention relative level change and the additional treated-control weekly slope change after intervention. Percent changes are on the modelled log scale transformed back to relative percentage differences.",
    )
    add_headline_table(doc, headline_rows)

    add_heading(doc, "Scheme-Level Reading", 1)
    add_para(
        doc,
        "For scheme 12d, active travel and pedestrian changes are not statistically clear. Cyclist counts show a positive but uncertain immediate level change, followed by a negative statistically flagged relative slope of about -0.15% per week. This suggests the cycling trajectory at treated countlines may have weakened relative to controls after the intervention month, but this remains exploratory.",
    )
    add_para(
        doc,
        "For scheme 12f, the active travel outcome shows a negative statistically flagged relative slope of about -0.11% per week, while pedestrian and cyclist outcomes are negative but uncertain. This does not currently indicate an increase in active travel uptake relative to controls.",
    )
    add_para(
        doc,
        "For scheme 13, all immediate level changes are negative but uncertain, and the post-intervention slope terms are not statistically clear. Cyclist counts have a positive but uncertain slope term, so the result should be treated as inconclusive rather than evidence of success or failure.",
    )

    add_heading(doc, "How This Should Be Written", 1)
    add_bullets(
        doc,
        [
            "Use cautious wording: relative trajectories, exploratory comparison, and evidence consistent with.",
            "Avoid definitive phrases such as the scheme caused a reduction or the scheme had no effect.",
            "Treat statistically flagged terms as signals requiring interpretation against sensor coverage, the first-day-of-installation-month dating assumption, seasonality, and site context.",
            "Use 12b and 12e to discuss data limitations and descriptive trends, not causal scheme effects.",
        ],
    )

    add_heading(doc, "Immediate Next Work", 1)
    add_bullets(
        doc,
        [
            "Manually verify shortlisted control countlines against local scheme maps and street-level context.",
            "Produce a deprivation/affluence comparison table for treated sites and valid controls using LSOA-linked IMD and Census indicators.",
            "Report pedestrian and cyclist outcomes separately first, then use active travel total as a summary measure.",
            "Write a limitations paragraph that explicitly addresses the month-level installation-date assumption, limited pre-intervention coverage, and control contamination risk.",
        ],
    )

    add_heading(doc, "Figures for Results Chapter", 1)
    add_para(
        doc,
        "The following plots are the current observed-versus-fitted model diagnostics. They are useful for the results chapter, but captions should describe them as exploratory model fits rather than final causal estimates.",
    )
    add_plot(
        doc,
        MODEL_DIR / "plots" / "model_observed_fitted_active_per_observed_day.png",
        "Figure 1. Active travel observed and fitted weekly counts.",
    )
    add_plot(
        doc,
        MODEL_DIR / "plots" / "model_observed_fitted_pedestrian_per_observed_day.png",
        "Figure 2. Pedestrian observed and fitted weekly counts.",
    )
    add_plot(
        doc,
        MODEL_DIR / "plots" / "model_observed_fitted_cyclist_per_observed_day.png",
        "Figure 3. Cyclist observed and fitted weekly counts.",
    )

    doc.save(OUT)


if __name__ == "__main__":
    build_doc()
    print(OUT)
